In [2]:
##Data Ingestion 
from langchain_core.documents import Document

In [3]:
doc=Document(
    page_content="this is the main text content I am using to create RAG",
    metadata={
        "source":"exmaple.txt",
        "pages":1,
        "author":"Vedashree",
        "date_created":"2025-01-01"
    }
)
doc

Document(metadata={'source': 'exmaple.txt', 'pages': 1, 'author': 'Vedashree', 'date_created': '2025-01-01'}, page_content='this is the main text content I am using to create RAG')

In [4]:
## Create a simple txt file
import os
os.makedirs("../data/text_files",exist_ok=True)

In [5]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}
for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [10]:
from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [11]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False

)

documents=dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popu

In [4]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

## load all the PDF files from the directory
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ## Pattern to match files  
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False

)

pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': '', 'creationdate': '2026-06-05T21:00:59+05:30', 'source': '..\\data\\pdf\\TCS.pdf', 'file_path': '..\\data\\pdf\\TCS.pdf', 'total_pages': 1, 'format': 'PDF 1.7', 'title': 'Gmail - TCS NQT for Priority Institutes Batch 2027_ 9th June 2026', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-06-05T21:00:59+05:30', 'trapped': '', 'modDate': "D:20260605210059+05'30'", 'creationDate': "D:20260605210059+05'30'", 'page': 0}, page_content='')]

In [12]:
type(pdf_documents[0])

langchain_core.documents.base.Document

In [25]:
from pathlib import Path
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: chunking_test_document.pdf
  ✓ Loaded 3 pages

Processing: TCS.pdf
  ✓ Loaded 1 pages

Total documents loaded: 4


In [26]:


from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def split_documents(documents, chunk_size=500, chunk_overlap=50):

    if not documents:
        print("No documents received!")
        return []

    # Convert raw strings into Document objects
    if isinstance(documents[0], str):
        documents = [
            Document(page_content=doc, metadata={})
            for doc in documents
        ]

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    print("Number of documents:", len(all_pdf_documents))

    print("Document content:")
    print(repr(all_pdf_documents[0].page_content))

    print("Content length:")
    print(len(all_pdf_documents[0].page_content))

    split_docs = text_splitter.split_documents(documents)

    print(f"Input documents: {len(documents)}")
    print(f"Total chunks created: {len(split_docs)}")

    for i, doc in enumerate(split_docs[:3]):
        print(f"\n--- Chunk {i+1} ---")
        print(f"Length: {len(doc.page_content)}")
        print(f"Content:\n{doc.page_content}")
        print(f"Metadata: {doc.metadata}")

    return split_docs


chunks = split_documents(all_pdf_documents)


Number of documents: 4
Document content:
"Page 1\n PDF TEXT EXTRACTION & CHUNKING TEST\nA sample document for testing PDF loaders, text extraction, and text chunking\nDocument Information\nDocument ID: CHUNK-TEST-001\nVersion: 1.0\nPrepared: November 2026\nPurpose: This document contains headings, paragraphs, lists, a table, and multiple sections so that a\nPDF-processing pipeline can be tested with different content structures.\n1. Introduction to Document Chunking\nDocument chunking is the process of splitting a long document into smaller pieces of text. These pieces are\ncommonly called chunks. Chunking is useful in retrieval-augmented generation (RAG) systems because a\nlanguage model can retrieve only the sections that are relevant to a user's question instead of processing an\nentire document every time.\nA good chunking strategy should preserve meaning. For example, a paragraph explaining a single concept\nshould ideally remain together. If a chunk ends in the middle of a senten